<a href="https://colab.research.google.com/github/ardizzone88/motor-inteligente-de-precios/blob/master/motor-inteligente-de-precios.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

# 🎯 Monitor Inteligente de Precios y Competencia
### Dynamic Pricing Analytics para Sellers de E-commerce

**Autor:** David | Data Analyst Portfolio Project
**Stack:** Python · Pandas · Plotly · Scikit-Learn (K-Means, Regresión Lineal) · Best Buy Products API

---

## 📋 El problema de negocio

Todos los días, miles de vendedores de e-commerce fijan un precio y se olvidan de él. Mientras tanto, la competencia:

- 📉 Baja precios agresivamente para liquidar stock
- 🚫 Se queda sin inventario y deja la cancha libre (oportunidad perdida si no la detectás a tiempo)
- 🎯 Ajusta su oferta varias veces por día con herramientas que el vendedor chico no puede pagar

Este notebook construye, de punta a punta, el motor analítico de un **producto SaaS de Dynamic Pricing**: un sistema que monitorea a la competencia, detecta quiebres de stock y bajadas de precio agresivas en tiempo real.

> ⚠️ **Nota sobre la fuente de datos:** la idea original de este proyecto era consumir la API pública de MercadoLibre. Sin embargo, **desde abril de 2025 MercadoLibre cerró el acceso público a su API**.
>
> Para que el proyecto siga funcionando con **datos reales y verificables hoy**, se usa la **[Best Buy Products API](https://developer.bestbuy.com/)**: una API pública, gratuita, con key instantáneo, sin OAuth.
>
> 💡 **Nota de transparencia metodológica:** la Best Buy API entrega el snapshot de precio/stock *actual* de cada producto, no su historial. Para poder analizar tendencias (bajadas de precio, quiebres, velocidad de reposición), este notebook **genera sintéticamente un historial de 14 días** anclado a datos reales del día 14 (hoy). Desde el análisis en adelante, el sistema funciona igual con datos reales históricos o sintéticos: detecta patterns, sugiere precios óptimos.

## ⚙️ 0. Setup del entorno

In [1]:
# Instalación de librerías (en Colab, plotly y sklearn ya vienen preinstalados)
!pip install requests --quiet

import requests
import pandas as pd
import numpy as np
import json
import time
from datetime import datetime, timedelta

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression

np.random.seed(42)
pd.set_option('display.max_columns', None)

print("Entorno listo ✅")


Entorno listo ✅


## 1️⃣ Recolección de datos — Best Buy Products API (real)

La [Best Buy Developer API](https://developer.bestbuy.com/) es pública y gratuita: te registrás con tu email y te entrega una API key al instante (sin OAuth, sin aprobación de partner, sin límite de consultas para desarrollo).

**Para correr esta celda con datos 100% reales:**
1. Entrá a https://developer.bestbuy.com/ → "Get API Key" (gratis, instantáneo).
2. Pegá tu key en la variable `BESTBUY_API_KEY` de la celda siguiente.

Si dejás la key en blanco (o la API no responde), el notebook cae automáticamente al generador sintético de la sección 2 — el resto del análisis funciona exactamente igual en ambos casos.

In [2]:
BESTBUY_API_KEY = ""  # 👈 Pegá aquí tu API key gratuita de https://developer.bestbuy.com/

CATEGORIA = "bluetooth headphones"  # Categoría/búsqueda real dentro del catálogo de Best Buy
LIMIT = 20

def consultar_bestbuy(query, api_key, limit=20):
    """Consulta la API real de Best Buy y devuelve un DataFrame con el snapshot actual de precio/stock."""
    if not api_key:
        print("ℹ️ No se configuró BESTBUY_API_KEY. Se usará un dataset sintético equivalente.")
        return None

    url = f"https://api.bestbuy.com/v1/products(search={query})"
    params = {
        "apiKey": api_key,
        "format": "json",
        "pageSize": limit,
        "show": "sku,name,manufacturer,regularPrice,salePrice,onSale,onlineAvailability,inStoreAvailability,customerReviewAverage,customerReviewCount",
    }

    try:
        resp = requests.get(url, params=params, timeout=10)
        resp.raise_for_status()
        data = resp.json()

        registros = []
        for item in data.get("products", []):
            registros.append({
                "sku": item.get("sku"),
                "titulo": item.get("name"),
                "marca": item.get("manufacturer"),
                "precio": item.get("salePrice", item.get("regularPrice")),
                "precio_regular": item.get("regularPrice"),
                "en_oferta": item.get("onSale"),
                "disponible_online": item.get("onlineAvailability"),
                "disponible_tienda": item.get("inStoreAvailability"),
                "rating": item.get("customerReviewAverage"),
            })

        df = pd.DataFrame(registros)
        print(f"✅ {len(df)} productos obtenidos en tiempo real desde la API de Best Buy")
        return df

    except Exception as e:
        print(f"⚠️ No se pudo consultar la API real ({e}). Se usará un dataset sintético equivalente.")
        return None

df_api = consultar_bestbuy(CATEGORIA, BESTBUY_API_KEY, LIMIT)
if df_api is not None:
    display(df_api.head(10))


ℹ️ No se configuró BESTBUY_API_KEY. Se usará un dataset sintético equivalente.


## 2️⃣ Generador de datos sintéticos (fallback + simulación de historial)

Usamos esta función en dos casos:
1. Si no se configuró una API key real, o la API no respondió (rate limit, sin conexión, cambios de esquema).
2. Siempre, para **simular los 14 días de historial** de precio y stock que la API no entrega, anclando el día 14 (hoy) a los precios reales si están disponibles.

In [4]:
import os

COMPETIDORES_DEFAULT = ["Sony", "JBL", "Bose", "Apple", "Beats", "Skullcandy"]

def generar_dataset_sintetico(df_real=None, dias=14, n_competidores=6):
    """Genera un historial de 14 días de precio/stock por competidor, realista y reproducible.
    Si hay datos reales de la API de Best Buy, usa las marcas y precios reales como ancla."""

    if df_real is not None and df_real["marca"].nunique() >= n_competidores:
        precios_reales = df_real.groupby("marca")["precio"].mean().sort_values(ascending=False)
        competidores = precios_reales.index[:n_competidores].tolist()
        precios_base = precios_reales.values[:n_competidores]
        print(f"📡 Usando {n_competidores} marcas reales (anclaje de precio real, vía API): {', '.join(competidores)}")
    else:
        competidores = COMPETIDORES_DEFAULT[:n_competidores]
        precios_base = np.random.uniform(40, 350, n_competidores)  # rango realista de auriculares en USD

    registros = []
    hoy = datetime.now()

    for i, competidor in enumerate(competidores):
        precio_actual = precios_base[i]
        stock_actual = np.random.randint(8, 40)
        # Cada competidor tiene una "personalidad": agresivo, estable o premium
        personalidad = np.random.choice(["agresivo", "estable", "premium"], p=[0.35, 0.4, 0.25])
        # Día "evento" garantizado por competidor agresivo: una liquidación fuerte de precio
        dia_liquidacion = np.random.randint(3, dias - 1) if personalidad == "agresivo" else None

        for d in range(dias):
            fecha = hoy - timedelta(days=(dias - 1 - d))

            # Variación de precio según personalidad
            if personalidad == "agresivo":
                shock = np.random.normal(-0.015, 0.05)
                if d == dia_liquidacion:
                    shock = -np.random.uniform(0.10, 0.18)  # liquidación: -10% a -18% en un día
            elif personalidad == "premium":
                shock = np.random.normal(0.005, 0.01)
            else:
                shock = np.random.normal(0, 0.02)

            precio_actual = max(precio_actual * (1 + shock), precios_base[i] * 0.5)

            # Ventas del día (más ventas si el precio es competitivo, y un pico tras liquidar)
            precio_rel = precio_actual / np.mean(precios_base)
            lam_ventas = max(0.5, 7 - precio_rel * 4.5)
            ventas_dia = np.random.poisson(lam_ventas)
            stock_actual = max(0, stock_actual - ventas_dia)

            # Reposición ocasional (no garantizada: así puede llegar a quiebre real de stock)
            if stock_actual < 4 and np.random.rand() < 0.25:
                stock_actual += np.random.randint(15, 40)

            registros.append({
                "fecha": fecha.date(),
                "competidor": competidor,
                "personalidad": personalidad,
                "precio": round(precio_actual, 2),
                "stock_disponible": stock_actual,
                "vendidos_dia": ventas_dia,
            })

    return pd.DataFrame(registros)

df_historico = generar_dataset_sintetico(df_api, dias=14, n_competidores=6)
os.makedirs('data', exist_ok=True) # Create the directory if it doesn't exist
df_historico.to_csv("data/historico_precios_competencia.csv", index=False)
df_historico.tail(10)

fecha,competidor,personalidad,precio,stock_disponible,vendidos_dia
74,Skullcandy,premium,63.42,12,7
75,Skullcandy,premium,63.14,10,2
76,Skullcandy,premium,63.66,4,6
77,Skullcandy,premium,63.18,0,5
78,Skullcandy,premium,63.16,0,5
79,Skullcandy,premium,63.72,37,6
80,Skullcandy,premium,64.65,33,4
81,Skullcandy,premium,65.32,30,3
82,Skullcandy,premium,65.06,22,8
83,Skullcandy,premium,64.79,19,3


## 3️⃣ Limpieza y preparación de variables

In [5]:
df = df_historico.copy()
df["fecha"] = pd.to_datetime(df["fecha"])
df = df.sort_values(["competidor", "fecha"]).reset_index(drop=True)

# Variación diaria de precio por competidor (clave para detectar bajadas agresivas)
df["precio_dia_anterior"] = df.groupby("competidor")["precio"].shift(1)
df["variacion_pct"] = ((df["precio"] - df["precio_dia_anterior"]) / df["precio_dia_anterior"]) * 100

# Acumulado de ventas
df["vendidos_acum"] = df.groupby("competidor")["vendidos_dia"].cumsum()

# Flag de quiebre de stock
df["quiebre_stock"] = df["stock_disponible"] == 0

print(f"Dataset limpio: {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"Competidores monitoreados: {df['competidor'].nunique()} | Período: {df['fecha'].min().date()} → {df['fecha'].max().date()}")
df.head()


Dataset limpio: 84 filas x 10 columnas
Competidores monitoreados: 6 | Período: 2026-06-05 → 2026-06-18


,fecha,competidor,personalidad,precio,stock_disponible,vendidos_dia,precio_dia_anterior,variacion_pct,vendidos_acum,quiebre_stock
0,2026-06-05,Apple,estable,125.58,6,2,NaN,NaN,2,False
1,2026-06-06,Apple,estable,121.41,6,0,125.58,-3.320592,2,False
2,2026-06-07,Apple,estable,119.32,4,2,121.41,-1.721440,4,False
3,2026-06-08,Apple,estable,118.40,0,5,119.32,-0.771036,9,True
4,2026-06-09,Apple,estable,119.77,22,5,118.40,1.157095,14,False


## 4️⃣ Panorama competitivo — Evolución de precios

Visualizamos cómo se movió cada competidor durante las dos últimas semanas.

In [6]:
fig = px.line(df, x="fecha", y="precio", color="competidor",
              title="Evolución de precios por competidor (14 días)",
              labels={"precio": "Precio ($)", "fecha": "Fecha"},
              markers=True)
fig.update_layout(template="plotly_white", legend_title="Marca", height=500)
fig.show()


## 5️⃣ Detección de quiebres de stock ⚠️

Cuando un competidor se queda sin stock, abre una ventana de oportunidad: la demanda no desaparece, se redirige. Detectarlo rápido permite ajustar precio o stock propio para capturarla.

In [7]:
quiebres = df[df["quiebre_stock"]].groupby("competidor").size().reset_index(name="dias_sin_stock")
quiebres = quiebres.sort_values("dias_sin_stock", ascending=False)

print("📦 Resumen de quiebres de stock en los últimos 14 días:")
print(quiebres.to_string(index=False) if len(quiebres) > 0 else "Sin quiebres de stock detectados en el período.")

fig = px.bar(df.groupby(["fecha"])["quiebre_stock"].sum().reset_index(),
             x="fecha", y="quiebre_stock",
             title="Competidores sin stock por día",
             labels={"quiebre_stock": "Cantidad de competidores sin stock"})
fig.update_layout(template="plotly_white", height=400)
fig.show()


📦 Resumen de quiebres de stock en los últimos 14 días:
competidor  dias_sin_stock
     Beats               5
Skullcandy               2
      Sony               2
     Apple               1


Plotly visualization

## 6️⃣ Detección de bajadas agresivas de precio 📉

Definimos como "bajada agresiva" cualquier caída de precio mayor al **8% en un solo día** respecto al día anterior — un umbral típico que indica liquidación de stock o reacción defensiva a la competencia.

In [8]:
bajadas_agresivas = df[(df["variacion_pct"] < -8)].copy()
bajadas_agresivas = bajadas_agresivas[["fecha", "competidor", "precio_dia_anterior", "precio", "variacion_pct"]].drop_duplicates()

print(f"🚨 Se detectaron {len(bajadas_agresivas)} bajadas agresivas de precio en el período:")
if len(bajadas_agresivas) > 0:
    display(bajadas_agresivas.round(2))


🚨 Se detectaron 1 bajadas agresivas de precio en el período:


,fecha,competidor,precio_dia_anterior,precio,variacion_pct
53,2026-06-16,JBL,143.35,119.07,-16.9
